# HM-RAG (End-to-End, Colab)

This notebook runs **ingestion (MMKG-style)** into **Qdrant + Neo4j**, then runs **inference** using **hosted OpenAI models** (no local GPU weights/Ollama).

## You will set these secrets in Colab (Environment/Secrets)
- `OPENAI_API_KEY`
- `SERPER_API_KEY` (optional for web retrieval)
- Qdrant: `QDRANT_HOST`, `QDRANT_PORT` (optional), `QDRANT_API_KEY`, and optionally `QDRANT_HTTPS=1`
- Neo4j: `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`

> If you use Qdrant Cloud, set `QDRANT_HTTPS=1` and set `QDRANT_HOST` to something like `xxxxxx.qdrant.tech`.


In [ ]:
# Secure API key loading (do not hardcode secrets in notebooks)
import os

def get_secret(name):
    """Load secrets from Colab secrets first, then environment variables."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name)

for _key in ['OPENAI_API_KEY', 'HF_TOKEN', 'SERPER_API_KEY', 'QDRANT_API_KEY', 'NEO4J_PASSWORD']:
    _value = get_secret(_key)
    if _value:
        os.environ[_key] = _value


In [ ]:
%pip -q install -U openai qdrant-client neo4j tqdm requests==2.32.4 transformers accelerate qwen-vl-utils "Pillow<12.0"

In [ ]:
%pip install requests==2.32.4

In [ ]:
# Fetch ScienceQA
%cd /kaggle/working
!rm -rf ScienceQA
!git clone -q https://github.com/lupantech/ScienceQA
%cd ScienceQA/
!bash tools/download.sh

In [ ]:
# Copy the auto-extracted HM-RAG package into Kaggle's writable directory
import os, sys, glob

target = '/kaggle/working/hmrag_full/'
if os.path.exists(target):
    !rm -rf /kaggle/working/hmrag_full

# Search for the hmrag_full directory inside Kaggle's inputs
source_dirs = glob.glob('/kaggle/input/**/hmrag_full', recursive=True)

if source_dirs:
    source = source_dirs[0]
    print(f"Found HM-RAG source at: {source}")
    !cp -r "{source}" /kaggle/working/
else:
    print("Could not find hmrag_full inside /kaggle/input. Let's list what's there:")
    !ls -la /kaggle/input/

# Make the package importable
sys.path.append(target)
print('Copied to:', target)

In [ ]:
import os, glob

# problems.json
cands = glob.glob('/kaggle/working/ScienceQA/**/problems.json', recursive=True)
assert cands, 'Could not find problems.json under /kaggle/working/ScienceQA'
PROBLEMS_JSON = cands[0]
print('PROBLEMS_JSON:', PROBLEMS_JSON)

IMAGE_ROOT = ''

# Strategy 1: Look for train/test/val directly in /kaggle/working
if os.path.isdir('/kaggle/working/train'):
    try:
        subdirs = os.listdir('/kaggle/working/train')
        if any(d.isdigit() for d in subdirs[:20]):
            sample_qid = next((d for d in subdirs if d.isdigit()), None)
            if sample_qid:
                test_img = f'/kaggle/working/train/{sample_qid}/image.png'
                if os.path.exists(test_img):
                    IMAGE_ROOT = '/kaggle/working'
                    print(f'Found IMAGE_ROOT at: {IMAGE_ROOT}')
    except Exception:
        pass

# Strategy 2: Search the entire /kaggle/working directory for train/test/val with images
if not IMAGE_ROOT:
    print("Searching /kaggle/working for train/test/val directories...")
    for root, dirs, files in os.walk('/kaggle/working'):
        dirs[:] = [d for d in dirs if not d.startswith('.') and d != 'sample_data']
        if 'train' in dirs:
            train_path = os.path.join(root, 'train')
            try:
                subdirs = os.listdir(train_path)
                has_numbered = any(d.isdigit() for d in subdirs[:20])
                if has_numbered:
                    sample_qid = next((d for d in subdirs if d.isdigit()), None)
                    if sample_qid:
                        test_img = os.path.join(train_path, sample_qid, 'image.png')
                        if os.path.exists(test_img):
                            IMAGE_ROOT = root
                            print(f'Found IMAGE_ROOT at: {IMAGE_ROOT}')
                            break
            except Exception:
                continue

# Strategy 3: Look for any image.png and work backwards
if not IMAGE_ROOT:
    print("Searching for any image.png files in /kaggle/working...")
    image_files = glob.glob('/kaggle/working/**/image.png', recursive=True)
    if image_files:
        sample = image_files[0]
        print(f'Sample image found: {sample}')
        parts = sample.split(os.sep)
        for split in ['train', 'test', 'val']:
            if split in parts:
                idx = parts.index(split)
                IMAGE_ROOT = os.sep.join(parts[:idx])
                print(f'Extracted IMAGE_ROOT: {IMAGE_ROOT}')
                break

# Strategy 4: Fallback
if not IMAGE_ROOT:
    print("Trying fallback based on problems.json location...")
    problems_dir = os.path.dirname(PROBLEMS_JSON)
    for candidate in [
        os.path.join(problems_dir, 'images'),
        os.path.join(problems_dir, '..', 'images'),
        os.path.join(problems_dir, '..', '..', 'images'),
        problems_dir, 
    ]:
        candidate = os.path.abspath(candidate)
        if os.path.isdir(candidate):
            if os.path.isdir(os.path.join(candidate, 'train')):
                IMAGE_ROOT = candidate
                print(f'Found IMAGE_ROOT at: {IMAGE_ROOT}')
                break

assert IMAGE_ROOT, (
    'Could not determine IMAGE_ROOT. Please manually set it.\n'
    'Run this to find images:\n'
    '  !find /kaggle/working -name "image.png" -type f | head -5'
)

print(f'\n✓ IMAGE_ROOT: {IMAGE_ROOT}')

print('\nVerifying split directories:')
for split_name in ['train', 'test', 'val']:
    split_dir = os.path.join(IMAGE_ROOT, split_name)
    if os.path.isdir(split_dir):
        try:
            contents = os.listdir(split_dir)
            n_subdirs = len([d for d in contents if os.path.isdir(os.path.join(split_dir, d))])
            n_images = len(glob.glob(os.path.join(split_dir, '*/image.png')))
            print(f'  ✓ {split_name}/: {n_subdirs} subdirectories, {n_images} images')
        except Exception as e:
            print(f'  ✗ {split_name}/: Error - {e}')
    else:
        print(f'  ✗ {split_name}/: NOT FOUND')

os.environ['PROBLEMS_JSON'] = PROBLEMS_JSON
os.environ['IMAGE_ROOT'] = IMAGE_ROOT

In [ ]:
# Read env vars (set these in Colab secrets)
import os

# You can set your API keys here directly or rely on the Colab environment variables/secrets.
# Uncomment and replace with your values if needed:
os.environ['OPENAI_API_KEY'] = get_secret('OPENAI_API_KEY') or ''
os.environ['SERPER_API_KEY']=get_secret('SERPER_API_KEY') or ''
os.environ['QDRANT_HOST'] = 'https://deed0e21-a677-4804-a024-f3528b4e3d6b.europe-west3-0.gcp.cloud.qdrant.io'
os.environ['QDRANT_PORT'] = '6333'
os.environ['QDRANT_API_KEY'] = get_secret('QDRANT_API_KEY') or ''
os.environ['QDRANT_COLLECTION'] = 'hmrag'
os.environ['NEO4J_URI'] = 'neo4j+s://43773969.databases.neo4j.io'
os.environ['NEO4J_USER'] = 'neo4j'
os.environ['NEO4J_PASSWORD'] = get_secret('NEO4J_PASSWORD') or ''
os.environ['HF_TOKEN'] = get_secret('HF_TOKEN') or ''

print('Environment variables configured.')

print('OK: env vars loaded')


In [ ]:
# Read env vars (set these in Colab secrets)
import os

# You can set your API keys here directly or rely on the Colab environment variables/secrets.
# Uncomment and replace with your values if needed:
os.environ['OPENAI_API_KEY'] = get_secret('OPENAI_API_KEY') or ''
#os.environ['SERPER_API_KEY'] = get_secret('SERPER_API_KEY') or ''
os.environ['QDRANT_HOST'] = 'https://deed0e21-a677-4804-a024-f3528b4e3d6b.europe-west3-0.gcp.cloud.qdrant.io'
os.environ['QDRANT_PORT'] = '6333'
os.environ['QDRANT_API_KEY'] = get_secret('QDRANT_API_KEY') or ''
os.environ['QDRANT_COLLECTION'] = 'hmrag'
os.environ['NEO4J_URI'] = 'neo4j+s://43773969.databases.neo4j.io'
os.environ['NEO4J_USER'] = 'neo4j'
os.environ['NEO4J_PASSWORD'] = get_secret('NEO4J_PASSWORD') or ''
os.environ['HF_TOKEN'] = get_secret('HF_TOKEN') or ''

print('Environment variables configured.')

print('OK: env vars loaded')


In [ ]:
import os

# ── Optional: disable graph ingestion (Neo4j writes) ──
# Set to False if you want to ingest into Neo4j again.
DISABLE_GRAPH_INGESTION = True

# ── Optional: disable graph ingestion (Neo4j writes) ──
if DISABLE_GRAPH_INGESTION:
    mmkg_path = '/kaggle/working/hmrag_full/ingest_scienceqa_mmkg.py'
    if os.path.exists(mmkg_path):
        with open(mmkg_path, 'r') as f:
            mmkg = f.read()

        # Comment out all Neo4j write calls (Problem/Chunk/KG). We keep embeddings + Qdrant upsert.
        replacements = {
            'upsert_neo4j_problem(neo4j_driver, qid, question, split, img_path, answer_idx)':
                '# upsert_neo4j_problem(neo4j_driver, qid, question, split, img_path, answer_idx)  # GRAPH INGESTION DISABLED',
            'upsert_neo4j_chunk(neo4j_driver, qid, cid, ch, source="fused", embedding=vec)':
                '# upsert_neo4j_chunk(neo4j_driver, qid, cid, ch, source="fused", embedding=vec)  # GRAPH INGESTION DISABLED',
            'upsert_neo4j_kg(neo4j_driver, qid, kg.entities, kg.relations,\n                            entity_embeddings=entity_embeddings)':
                '# upsert_neo4j_kg(neo4j_driver, qid, kg.entities, kg.relations,\n#                            entity_embeddings=entity_embeddings)  # GRAPH INGESTION DISABLED',
        }

        changed = False
        for old, new in replacements.items():
            if old in mmkg:
                mmkg = mmkg.replace(old, new)
                changed = True

        # Also skip index creation since it hits Neo4j.
        if 'ensure_neo4j_indexes(neo4j_driver)' in mmkg:
            mmkg = mmkg.replace(
                'ensure_neo4j_indexes(neo4j_driver)',
                '# ensure_neo4j_indexes(neo4j_driver)  # GRAPH INGESTION DISABLED'
            )
            changed = True

        if changed:
            with open(mmkg_path, 'w') as f:
                f.write(mmkg)
            print('✓ Neo4j graph ingestion DISABLED in ingest_scienceqa_mmkg.py (Qdrant ingestion unaffected).')
        elif 'GRAPH INGESTION DISABLED' in mmkg:
            print('✓ Neo4j graph ingestion already disabled.')
        else:
            print('⚠ Could not patch ingest_scienceqa_mmkg.py to disable graph ingestion (pattern mismatch).')
    else:
        print(f'⚠ {mmkg_path} not found — cannot disable graph ingestion.')


# ── Fix 1: vlm_qwen imports in ingest_scienceqa_mmkg.py ──
file_path = '/kaggle/working/hmrag_full/ingest_scienceqa_mmkg.py'
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    with open(file_path, 'w') as f:
        for line in lines:
            if "from vlm_qwen import _load_qwen_vl" in line:
                f.write("        from hmrag_full.vlm_qwen import _load_qwen_vl\n")
            elif "from .vlm_qwen import describe_image_qwen" in line or "from hmrag_full.vlm_qwen import describe_image_qwen" in line:
                # Maintain original indentation so it doesn't break the try/except block
                indent = line[:len(line) - len(line.lstrip())]
                if not indent:  # fallback if stripped entirely previously
                    indent = "    "
                f.write(f"{indent}from hmrag_full.vlm_qwen import describe_image_qwen\n")
            elif 'choices=["openai", "hf_endpoint", "qwen"]' in line:
                f.write(line.replace('choices=["openai", "hf_endpoint", "qwen"]', 'choices=["openai", "hf_endpoint", "qwen", "smolvlm"]'))
            else:
                f.write(line)
    print(f"✓ Fixed argparse choices and vlm_qwen imports")

# ── Fix 2: prompt_format "QCM" → "QCM-A" in config.py ──
cfg_path = '/kaggle/working/hmrag_full/config.py'
if os.path.exists(cfg_path):
    with open(cfg_path, 'r') as f:
        content = f.read()
    if 'prompt_format: str = "QCM"' in content:
        content = content.replace('prompt_format: str = "QCM"', 'prompt_format: str = "QCM-A"')
        with open(cfg_path, 'w') as f:
            f.write(content)
        print(f"✓ Fixed prompt_format QCM → QCM-A")

# ── Fix 3: Serper URL (serper.dev → google.serper.dev) in web_retrieval.py ──
wr_path = '/kaggle/working/hmrag_full/retrieval/web_retrieval.py'
if os.path.exists(wr_path):
    with open(wr_path, 'r') as f:
        content = f.read()
    if 'https://serper.dev/search' in content:
        content = content.replace('https://serper.dev/search', 'https://google.serper.dev/search')
        with open(wr_path, 'w') as f:
            f.write(content)
        print(f"✓ Fixed Serper URL")

# ── Fix 4: Neo4j session.run(**params) → session.run(cypher, params) ──
gr_path = '/kaggle/working/hmrag_full/retrieval/graph_retrieval.py'
if os.path.exists(gr_path):
    with open(gr_path, 'r') as f:
        content = f.read()
    if 'session.run(cypher, **params)' in content:
        content = content.replace('session.run(cypher, **params)', 'session.run(cypher, params)')
        with open(gr_path, 'w') as f:
            f.write(content)
        print(f"✓ Fixed Neo4j session.run params")

# ── Fix 5: summary_agent_hosted.py regex for answer extraction ──
sa_path = '/kaggle/working/hmrag_full/summary_agent_hosted.py'
if os.path.exists(sa_path):
    with open(sa_path, 'r') as f:
        content = f.read()
    old_regex = r'''    def _get_result_letter(self, output: str) -> str:
        m = re.search(r"The answer is\s+([A-E])", output)
        return m.group(1) if m else "FAILED"'''
    new_regex = r'''    def _get_result_letter(self, output: str) -> str:
        m = re.search(r"[Tt]he answer is\\s+([A-Ea-e])", output)
        if m:
            return m.group(1).upper()
        m2 = re.search(r"Answer:\\s*([A-Ea-e])\\b", output)
        if m2:
            return m2.group(1).upper()
        return "FAILED"'''
    if old_regex in content:
        content = content.replace(old_regex, new_regex)
        with open(sa_path, 'w') as f:
            f.write(content)
        print(f"✓ Fixed answer regex")

print("\nAll patches applied. You can now re-run the ingestion and inference cells.")

Now that the `ingest_scienceqa_mmkg.py` file has been corrected on disk, please **re-run cell `bb5ec2c1`** to perform the ingestion.

In [ ]:
import os

# Ingest (MMKG-style): VLM image->text + entity/relation extraction into Neo4j + embeddings into Qdrant
# Tip: start with --max-items 50 to validate connectivity.

qdrant_host = os.environ["QDRANT_HOST"]
if qdrant_host.startswith('http://'):
    qdrant_host = qdrant_host.replace('http://', '')
elif qdrant_host.startswith('https://'):
    qdrant_host = qdrant_host.replace('https://', '')

qdrant_port = os.environ.get("QDRANT_PORT", "6333")
qdrant_https_flag = "--qdrant-https" if os.environ.get("QDRANT_HTTPS") == "1" else ""

# Auto-detect Qdrant Cloud usage and override port and https flag
if "cloud.qdrant.io" in qdrant_host or "qdrant.tech" in qdrant_host:
    qdrant_port = "443"
    qdrant_https_flag = "--qdrant-https"

# We run this from /kaggle/working so "hmrag_full" is recognized as a package.
cmd = f'''cd /kaggle/working && python -u -m hmrag_full.ingest_scienceqa_mmkg \\
  --dataset "{os.environ["PROBLEMS_JSON"]}" \\
  --image-root "{os.environ["IMAGE_ROOT"]}" \\
  --split train \\
  --max-items 500 \\
  --qdrant-host "{qdrant_host}" \\
  --qdrant-port "{qdrant_port}" \\
  --qdrant-api-key "{os.environ["QDRANT_API_KEY"]}" \\
  {qdrant_https_flag} \\
  --neo4j-uri "{os.environ["NEO4J_URI"]}" \\
  --neo4j-user "{os.environ["NEO4J_USER"]}" \\
  --neo4j-password "{os.environ["NEO4J_PASSWORD"]}" \\
  --openai-api-key "{os.environ["OPENAI_API_KEY"]}" \\
  --embedding-model "text-embedding-3-small" \\
  --use-vlm \\
  --vlm-provider smolvlm \\
  --vlm-model "HuggingFaceTB/SmolVLM-Instruct" \\
  --kg-model "gpt-4o-mini"
'''

# By using ! with the -u flag for python (unbuffered), the logs will print to the cell output in real-time.
!{cmd}

In [ ]:
# Run inference (hosted pipeline)
# Choose a qid that exists in your problems.json. Example: pick the first key.
import json

with open(PROBLEMS_JSON, 'r', encoding='utf-8') as f:
    problems = json.load(f)
qid = next(iter(problems.keys()))
print('Using qid:', qid)


In [ ]:
import json
import os
import re
from pathlib import Path
import subprocess
from tqdm import tqdm

# ── Disable graph retrieval (Neo4j) for inference ──
mra_path = '/kaggle/working/hmrag_full/multi_retrieval_agent_hosted.py'
if os.path.exists(mra_path):
    with open(mra_path, 'r') as f:
        content = f.read()
    
    # We alter the graph_retrieval find_top_k call to return a mock empty string
    # so the pipeline continues with only vector + web search
    old_graph_call = 'graph_response = self.graph_retrieval.find_top_k(question)'
    new_graph_call = 'graph_response = "Graph retrieval disabled by user request." # self.graph_retrieval.find_top_k(question)'
    
    if old_graph_call in content:
        content = content.replace(old_graph_call, new_graph_call)
        with open(mra_path, 'w') as f:
            f.write(content)
        print("✓ Graph retrieval DISABLED in multi_retrieval_agent_hosted.py\n")
    elif new_graph_call in content:
        print("✓ Graph retrieval already disabled.\n")
    else:
        print("⚠ Could not disable graph retrieval - pattern not found.\n")

# Fix QDRANT_HOST if it contains a protocol
qdrant_host = os.environ.get('QDRANT_HOST', '')
if qdrant_host.startswith('http://'):
    qdrant_host = qdrant_host.replace('http://', '')
elif qdrant_host.startswith('https://'):
    qdrant_host = qdrant_host.replace('https://', '')
os.environ['QDRANT_HOST'] = qdrant_host

qdrant_port = os.environ.get("QDRANT_PORT", "6333")
qdrant_https = os.environ.get("QDRANT_HTTPS") == "1"

# Auto-detect Qdrant Cloud usage and override port and https flag
if "cloud.qdrant.io" in qdrant_host or "qdrant.tech" in qdrant_host:
    qdrant_port = "443"
    qdrant_https = True

# Load problems
with open(PROBLEMS_JSON, 'r', encoding='utf-8') as f:
    problems = json.load(f)

# Get first 500 train and first 300 test questions
train_qids = [qid for qid, p in problems.items() if p.get('split') == 'train'][:500]
test_qids = [qid for qid, p in problems.items() if p.get('split') == 'test'][:300]

all_qids = train_qids + test_qids

print(f"Total questions to evaluate: {len(all_qids)}")
print(f"  - Train: {len(train_qids)}")
print(f"  - Test: {len(test_qids)}")

# Create results directory
results_dir = Path('/kaggle/working/results')
results_dir.mkdir(exist_ok=True)

# Store results
results = []

print("\n" + "="*80)
print("Starting inference...")
print("="*80 + "\n")

for idx, qid in enumerate(tqdm(all_qids, desc="Processing questions"), 1):
    problem = problems[qid]
    split = problem.get('split', 'unknown')

    # Build command
    cmd = [
        'python', '-m', 'hmrag_full.run_pipeline_hosted',
        '--dataset', PROBLEMS_JSON,
        '--qid', qid,
        '--qdrant-host', os.environ['QDRANT_HOST'],
        '--qdrant-port', qdrant_port,
        '--qdrant-api-key', os.environ.get('QDRANT_API_KEY', ''),
        # Ingestion used the script's default ('hmrag_collection'), but env var has 'hmrag'
        '--qdrant-collection', 'hmrag_collection',
    ]
    if qdrant_https:
        cmd.append('--qdrant-https')
    cmd += [
        '--neo4j-uri', os.environ['NEO4J_URI'],
        '--neo4j-user', os.environ['NEO4J_USER'],
        '--neo4j-password', os.environ['NEO4J_PASSWORD'],
        '--serper-api-key', os.environ.get('SERPER_API_KEY', ''),
        '--openai-api-key', os.environ['OPENAI_API_KEY'],
        '--image-root', IMAGE_ROOT,
        '--top-k', '5',
    ]

    # Run inference
    try:
        result = subprocess.run(
            cmd,
            cwd='/kaggle/working',
            capture_output=True,
            text=True,
            timeout=180,
            env={**os.environ, 'PYTHONPATH': '/kaggle/working'},
        )

        output = result.stdout + result.stderr

        # Parse predicted answer from the pipeline's actual output format
        predicted_answer = None

        m = re.search(r'Predicted answer index:\s*(\d+)', output)
        if m:
            predicted_answer = int(m.group(1))

        if predicted_answer is None:
            m = re.search(r'final answer:\s*(\d+)', output, re.IGNORECASE)
            if m:
                predicted_answer = int(m.group(1))
            else:
                m = re.search(r'final answer:\s*([A-E])', output, re.IGNORECASE)
                if m:
                    predicted_answer = ord(m.group(1).upper()) - ord('A')

        result_data = {
            'qid': qid,
            'split': split,
            'question': problem.get('question', ''),
            'correct_answer': problem.get('answer', 0),
            'predicted_answer': predicted_answer,
            'choices': problem.get('choices', []),
            'has_image': bool(problem.get('image', '')),
            'output': output[-2000:] if len(output) > 2000 else output,
            'success': result.returncode == 0,
            'returncode': result.returncode,
        }
        results.append(result_data)

        # Print progress
        status = "✓" if result.returncode == 0 else "✗"
        correct_marker = ""
        if predicted_answer is not None:
            is_correct = predicted_answer == problem.get('answer', 0)
            correct_marker = " [CORRECT]" if is_correct else " [WRONG]"
        elif result.returncode != 0:
            err_snippet = result.stderr.strip().split('\n')[-3:]
            correct_marker = f" [ERROR: {'|'.join(err_snippet)[-80:]}]"

        tqdm.write(f"{status} [{idx}/{len(all_qids)}] {split.upper()} qid={qid}{correct_marker}")

    except subprocess.TimeoutExpired:
        tqdm.write(f"✗ [{idx}/{len(all_qids)}] {split.upper()} qid={qid} [TIMEOUT]")
        results.append({
            'qid': qid,
            'split': split,
            'question': problem.get('question', ''),
            'correct_answer': problem.get('answer', 0),
            'error': 'timeout',
            'success': False,
        })
    except Exception as e:
        tqdm.write(f"✗ [{idx}/{len(all_qids)}] {split.upper()} qid={qid} [ERROR: {str(e)[:50]}]")
        results.append({
            'qid': qid,
            'split': split,
            'question': problem.get('question', ''),
            'correct_answer': problem.get('answer', 0),
            'error': str(e),
            'success': False,
        })

# Save results
results_file = results_dir / 'inference_results_500train_300test.json'
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2)

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)
print(f"\nResults saved to: {results_file}")

# Calculate statistics
train_results = [r for r in results if r.get('split') == 'train']
test_results = [r for r in results if r.get('split') == 'test']

def calc_accuracy(result_list):
    with_pred = [r for r in result_list if r.get('predicted_answer') is not None]
    if not with_pred:
        return 0, 0, 0
    correct = sum(1 for r in with_pred if r['predicted_answer'] == r['correct_answer'])
    return correct, len(with_pred), (correct / len(with_pred) * 100) if with_pred else 0

print(f"\n--- TRAIN SET ({len(train_results)} questions) ---")
train_correct, train_total, train_acc = calc_accuracy(train_results)
print(f"Accuracy: {train_correct}/{train_total} = {train_acc:.2f}%")
print(f"Successful runs: {sum(1 for r in train_results if r.get('success', False))}/{len(train_results)}")

print(f"\n--- TEST SET ({len(test_results)} questions) ---")
test_correct, test_total, test_acc = calc_accuracy(test_results)
print(f"Accuracy: {test_correct}/{test_total} = {test_acc:.2f}%")
print(f"Successful runs: {sum(1 for r in test_results if r.get('success', False))}/{len(test_results)}")

print(f"\n--- OVERALL ({len(results)} questions) ---")
all_correct, all_total, all_acc = calc_accuracy(results)
print(f"Accuracy: {all_correct}/{all_total} = {all_acc:.2f}%")
print(f"Successful runs: {sum(1 for r in results if r.get('success', False))}/{len(results)}")

In [ ]:
# Verify vector_retrieval.py has the correct Qdrant API fallback
# (The file already supports query_points → search_points → search)
import os
file_path = '/content/hmrag_full/hmrag_full/retrieval/vector_retrieval.py'
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()
    if 'query_points' in content:
        print(f"✓ {file_path} already has query_points fallback — no patch needed.")
    else:
        print(f"⚠ {file_path} may need updating — check qdrant-client version.")
else:
    print(f"⚠ File not found at {file_path} — skipping check.")


The `vector_retrieval.py` file has been updated to use `self.client.query` instead of `self.client.search`. Please **re-run cell `962cb1c2`** now to continue the inference process.